# XGBoost 모델 구성 설명

## 목표와 Target

- 목표: 학생별로 **해당 예측 주차 종료 시점에서 다음 7일 안의 이탈확률**을 예측합니다.
- 사용 데이터: `used_data/weekly_next_week_with_vle.csv`
- Target: `target_next_week_withdrawn` — 다음 7일 안에 이탈하면 1, 아니면 0
- 비교 범위: demo2가 보유한 4·19·25주차

## 사용 Feature

모델 입력은 `id_student`와 Target을 제외한 **98개 Feature**입니다. `id_student`는 모델 입력에 넣지 않고 GroupKFold 분할 기준으로만 사용합니다.

- 과목·운영·주차 정보
- 학생 기본 특성과 등록 시점 정보
- 현재·이전·누적 VLE 클릭과 활동 변화량
- 평가 제출·미제출·지각·점수 관련 정보

기존 최종 이탈 `target`과 `date_unregistration`은 Feature에서 제거했습니다.

## 전처리·검증

- 문자형 8개 Feature는 `OneHotEncoder(handle_unknown='ignore')`로 숫자로 바꿉니다.
- One-Hot Encoder는 각 Train Fold에만 학습하므로 Test 정보를 미리 보지 않습니다.
- `id_student` 기준 **3-Fold GroupKFold**로 같은 학생이 Train과 Test에 섞이지 않게 합니다.
- 각 Fold의 전처리기와 XGBoost 모델을 `models/xgboost_weekly_next_week`에 저장합니다.

## 하이퍼파라미터

| 항목 | 설정값 | 쉬운 의미 |
|---|---:|---|
| `n_estimators` | 500 | 최대 500개의 트리를 만듭니다. |
| `learning_rate` | 0.05 | 한 번에 조금씩 학습합니다. |
| `max_depth` | 6 | 트리를 지나치게 복잡하게 만들지 않습니다. |
| `subsample` | 0.8 | 매번 학습 행의 80%를 사용합니다. |
| `colsample_bytree` | 0.8 | 매 트리에서 Feature의 80%를 사용합니다. |
| `scale_pos_weight` | Fold별 자동 계산 | 매우 적은 이탈자를 더 중요하게 학습합니다. |
| `early_stopping_rounds` | 40 | 성능 개선이 멈추면 학습을 일찍 끝냅니다. |

## 결과를 읽는 순서

1. `Recall@Top-20%`: 위험 상위 20%가 실제 이탈자를 얼마나 포함하는지 확인합니다.
2. `PR-AUC`: 이탈자를 구분하는 전체적인 성능을 확인합니다.
3. `Brier`, `ECE`: 화면에 표시할 확률 자체가 실제 비율과 가까운지 확인합니다.

`scale_pos_weight`는 이탈자를 찾는 데 도움을 주지만 원시 확률을 높게 만들 수 있습니다. 따라서 Brier와 ECE가 나쁘면 원시 확률을 사용자 화면에 바로 표시하지 않습니다.


# XGBoost: 주차별 다음 주 이탈 예측

아래 첫 번째 코드 셀을 실행하면 학습과 평가가 진행됩니다. 설정을 바꾸려면 `01_xgboost_weekly_next_week.py` 상단의 **사용자가 직접 바꿀 수 있는 설정 구간**만 수정하면 됩니다.


In [ ]:
import runpy
runpy.run_path('01_xgboost_weekly_next_week.py', run_name='__main__')

In [ ]:
import pandas as pd
pd.read_csv('xgboost_weekly_next_week_metrics.csv')